# Hadoop Iceberg Catalog With On-Demand Cloud SQL Catalog

This notebook starts one Spark session with a GCS-backed Hadoop Iceberg catalog enabled by default.

Cloud SQL/JDBC catalog support is added only when you explicitly call `enable_cloudsql_catalog()`. That lets you keep Cloud SQL stopped for normal exploration, then query both catalogs in the same Spark session when Cloud SQL Auth Proxy is running.


## 1. Configuration

Defaults use the local service account key and Hadoop 3 GCS connector created for laptop Spark access.


In [ ]:
!../scripts/setup_notebook_gcs_prereqs.sh

In [ ]:
import os
import socket
import subprocess
from datetime import datetime, timezone
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "project-66783f65-9c3e-4880-9a3")
BUCKET = os.environ.get("GCS_ICEBERG_BUCKET", f"{PROJECT_ID}-alpaca-iceberg-warehouse")
HADOOP_WAREHOUSE = os.environ.get("GCS_HADOOP_WAREHOUSE", f"gs://{BUCKET}")

GCS_CONNECTOR_JAR = os.environ.get("GCS_CONNECTOR_JAR", "/tmp/gcs-connector-hadoop3-2.2.30-shaded.jar")
GCS_SERVICE_ACCOUNT_JSON = os.environ.get("GCS_SERVICE_ACCOUNT_JSON", "/tmp/alpaca-spark-gcs-reader.json")

HADOOP_CATALOG = os.environ.get("HADOOP_ICEBERG_CATALOG", "gcs_iceberg")
EXPLORE_NAMESPACE = os.environ.get("EXPLORE_NAMESPACE", "explore")
EXPLORE_TABLE = os.environ.get("EXPLORE_TABLE", "sample_bars")
EXPLORE_TABLE_ID = f"{HADOOP_CATALOG}.{EXPLORE_NAMESPACE}.{EXPLORE_TABLE}"

PROD_NAMESPACE = os.environ.get("ICEBERG_NAMESPACE", "alpaca")
PROD_TABLE = os.environ.get("ICEBERG_TABLE", "bars")

CLOUDSQL_CATALOG = os.environ.get("CLOUDSQL_ICEBERG_CATALOG", "alpaca_catalog")
CLOUDSQL_TABLE_ID = f"{CLOUDSQL_CATALOG}.{PROD_NAMESPACE}.{PROD_TABLE}"
CLOUDSQL_INSTANCE_CONNECTION_NAME = os.environ.get(
    "CLOUDSQL_INSTANCE_CONNECTION_NAME",
    "project-66783f65-9c3e-4880-9a3:us-east1:alpaca-iceberg-catalog",
)
CLOUDSQL_HOST = os.environ.get("CLOUDSQL_HOST", "127.0.0.1")
CLOUDSQL_PORT = int(os.environ.get("CLOUDSQL_PORT", "5432"))
CLOUDSQL_DB = os.environ.get("CLOUDSQL_DB", "iceberg")
CLOUDSQL_USER = os.environ.get("CLOUDSQL_USER", "iceberg")
CLOUDSQL_PASSWORD = os.environ.get("CLOUDSQL_PASSWORD")
CLOUDSQL_SECRET_NAME = os.environ.get("CLOUDSQL_SECRET_NAME", "ICEBERG_DB_PASSWORD")

os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print(f"Project          : {PROJECT_ID}")
print(f"Warehouse        : {HADOOP_WAREHOUSE}")
print(f"Hadoop catalog   : {HADOOP_CATALOG}")
print(f"Explore table    : {EXPLORE_TABLE_ID}")
print(f"Cloud SQL catalog: {CLOUDSQL_CATALOG} (disabled until enable_cloudsql_catalog() is called)")
print(f"GCS key exists   : {Path(GCS_SERVICE_ACCOUNT_JSON).exists()} ({GCS_SERVICE_ACCOUNT_JSON})")
print(f"GCS jar exists   : {Path(GCS_CONNECTOR_JAR).exists()} ({GCS_CONNECTOR_JAR})")


## 2. Start Spark With HadoopCatalog

This is the default path. It does not require Cloud SQL.


In [ ]:
if not Path(GCS_CONNECTOR_JAR).is_file():
    raise RuntimeError(f"Missing GCS connector jar: {GCS_CONNECTOR_JAR}")
if not Path(GCS_SERVICE_ACCOUNT_JSON).is_file():
    raise RuntimeError(f"Missing service account key: {GCS_SERVICE_ACCOUNT_JSON}")

SPARK_PACKAGES = os.environ.get(
    "SPARK_PACKAGES",
    ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.2",
        "org.postgresql:postgresql:42.7.7",
    ]),
)

spark = (
    SparkSession.builder
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .appName("alpaca-hadoop-catalog-on-demand-cloudsql")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.caseSensitive", "true")
    .config("spark.ui.enabled", os.environ.get("SPARK_UI_ENABLED", "false"))
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
    .config("spark.jars", GCS_CONNECTOR_JAR)
    .config("spark.jars.packages", SPARK_PACKAGES)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.fs.gs.project.id", PROJECT_ID)
    .config("spark.hadoop.fs.gs.auth.type", "SERVICE_ACCOUNT_JSON_KEYFILE")
    .config("spark.hadoop.fs.gs.auth.service.account.enable", "true")
    .config("spark.hadoop.fs.gs.auth.service.account.json.keyfile", GCS_SERVICE_ACCOUNT_JSON)
    .config(f"spark.sql.catalog.{HADOOP_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{HADOOP_CATALOG}.type", "hadoop")
    .config(f"spark.sql.catalog.{HADOOP_CATALOG}.warehouse", HADOOP_WAREHOUSE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Enabled by default: {HADOOP_CATALOG}")


## 3. HadoopCatalog Exploration Read/Write

This creates an exploratory Iceberg table under the Hadoop catalog. It is separate from the production `alpaca.bars` table.


In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {HADOOP_CATALOG}.{EXPLORE_NAMESPACE}")
spark.sql(f"SHOW NAMESPACES IN {HADOOP_CATALOG}").show(truncate=False)
spark.sql(f"SHOW TABLES IN {HADOOP_CATALOG}.{EXPLORE_NAMESPACE}").show(truncate=False)


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {EXPLORE_TABLE_ID} (
        symbol STRING,
        t TIMESTAMP,
        close DOUBLE,
        source STRING
    )
    USING iceberg
    PARTITIONED BY (days(t))
""")

now_utc = datetime.now(timezone.utc).replace(microsecond=0)
rows = [
    ("AAPL", now_utc.isoformat(), 210.50, "notebook"),
    ("NVDA", now_utc.isoformat(), 125.25, "notebook"),
    ("TSLA", now_utc.isoformat(), 305.75, "notebook"),
]

df = spark.createDataFrame(rows, "symbol string, t string, close double, source string").withColumn("t", F.to_timestamp("t"))
df.writeTo(EXPLORE_TABLE_ID).append()

spark.sql(f"SELECT * FROM {EXPLORE_TABLE_ID} ORDER BY t DESC, symbol LIMIT 20").show(20, truncate=False)


In [ ]:
spark.sql(f"select * from {EXPLORE_TABLE_ID} limit 10").show()

## 4. Add Cloud SQL Catalog On Demand

Cloud SQL is not configured at Spark startup. Call `enable_cloudsql_catalog()` only after the Cloud SQL instance is up and Cloud SQL Auth Proxy is listening locally.

Proxy example:

```bash
cloud-sql-proxy project-66783f65-9c3e-4880-9a3:us-east1:alpaca-iceberg-catalog --port 5432
```


In [ ]:
def gcloud(*args: str) -> str:
    return subprocess.check_output(["gcloud", *args], text=True).strip()


def cloudsql_password() -> str:
    if CLOUDSQL_PASSWORD:
        return CLOUDSQL_PASSWORD
    return gcloud("secrets", "versions", "access", "latest", f"--secret={CLOUDSQL_SECRET_NAME}")


def assert_cloudsql_proxy() -> None:
    try:
        with socket.create_connection((CLOUDSQL_HOST, CLOUDSQL_PORT), timeout=3):
            return
    except OSError:
        message = (
            f"Cloud SQL Auth Proxy is not listening on {CLOUDSQL_HOST}:{CLOUDSQL_PORT}."
            + "\n\nStart it with:\n"
            + f"cloud-sql-proxy {CLOUDSQL_INSTANCE_CONNECTION_NAME} --port {CLOUDSQL_PORT}"
        )
        raise RuntimeError(message) from None


def enable_cloudsql_catalog() -> str:
    """Register the Cloud SQL JDBC Iceberg catalog in the existing Spark session."""
    assert_cloudsql_proxy()
    jdbc_uri = f"jdbc:postgresql://{CLOUDSQL_HOST}:{CLOUDSQL_PORT}/{CLOUDSQL_DB}"
    password = cloudsql_password()

    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}", "org.apache.iceberg.spark.SparkCatalog")
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog")
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.uri", jdbc_uri)
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.user", CLOUDSQL_USER)
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.password", password)
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.jdbc.schema-version", "V1")
    spark.conf.set(f"spark.sql.catalog.{CLOUDSQL_CATALOG}.warehouse", HADOOP_WAREHOUSE)

    print(f"Enabled Cloud SQL catalog: {CLOUDSQL_CATALOG}")
    return CLOUDSQL_CATALOG


def show_cloudsql_catalog() -> None:
    spark.sql(f"SHOW NAMESPACES IN {CLOUDSQL_CATALOG}").show(truncate=False)
    spark.sql(f"SHOW TABLES IN {CLOUDSQL_CATALOG}.{PROD_NAMESPACE}").show(truncate=False)

print("Cloud SQL helper functions loaded. Call enable_cloudsql_catalog() when the proxy is running.")


In [ ]:
enable_cloudsql_catalog()

## 5. Query Both Catalogs In The Same Session

Run this cell after calling `enable_cloudsql_catalog()`.


In [ ]:
spark.sql(f"select count(*) from alpaca_catalog.alpaca.bars").show()

In [ ]:
# Uncomment these lines after Cloud SQL Auth Proxy is running.
# enable_cloudsql_catalog()
# show_cloudsql_catalog()

# HadoopCatalog table query: always available after Spark starts.
# spark.sql(f"""
#     SELECT 'hadoop' AS catalog, symbol, COUNT(*) AS rows, MAX(t) AS latest_t
#     FROM {EXPLORE_TABLE_ID}
#     GROUP BY symbol
#     ORDER BY symbol
# """).show(truncate=False)

# Cloud SQL catalog query: available after enable_cloudsql_catalog().
spark.sql(f"""
    SELECT 'cloudsql' AS catalog, S AS symbol, COUNT(*) AS rows, MAX(t) AS latest_t
    FROM {CLOUDSQL_TABLE_ID}
    GROUP BY S
    ORDER BY rows DESC, symbol
    LIMIT 20
""").show(truncate=False)


In [ ]:
CLOUDSQL_TABLE_ID

In [ ]:
spark.sql(
    """
    create table gcs_iceberg.explore.test_bars as 
    select * from alpaca_catalog.alpaca.bars
"""
)

## 6. Cleanup Optional Exploration Table

Keep this commented unless you intentionally want to remove the exploratory Iceberg table.


In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {EXPLORE_TABLE_ID}")
# spark.sql(f"DROP NAMESPACE IF EXISTS {HADOOP_CATALOG}.{EXPLORE_NAMESPACE}")


## 7. Stop Spark


In [ ]:
# spark.stop()
